# SurveyGPT – a Surveying assistant for Civil Engineering

**What it does:** answers theory questions and solves numerical problems in Surveying
(chain/tape, compass, levelling, theodolite & traverse, tacheometry, curves, area computation).

**How it works:** Gemini (LLM) + your own course notes (RAG) + Python calculator tools (for reliable arithmetic) + a Gradio chat window.

**Run order:** run every cell from top to bottom. Before starting, add your Gemini API key in
Colab → left sidebar → 🔑 *Secrets* → name `GEMINI_API_KEY` → turn on *Notebook access*.
Get a free key at https://aistudio.google.com/app/apikey

In [ ]:
!pip install -q -U google-genai gradio sentence-transformers pypdf numpy

In [ ]:
# Install system-level OCR tools and Python libraries for image reading
!apt-get install -y tesseract-ocr poppler-utils
!pip install pdf2image pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
poppler-utils is already the newest version (24.02.0-1ubuntu9.9).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


The cell above installed `tesseract-ocr` (a command-line OCR engine) and `pdf2image` (a Python library to convert PDF pages into images) along with `pytesseract` (a Python wrapper for Tesseract). These are necessary for the `load_chunks` function to handle image-based PDF pages.

In [ ]:
import os
# ---- Load the API key safely (never paste keys directly into code) ----
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    import getpass
    GEMINI_API_KEY = getpass.getpass("Paste your Gemini API key: ")
print("API key loaded:", bool(GEMINI_API_KEY))

API key loaded: True


## Calculator tools
LLMs are unreliable at long arithmetic, so the numerical work is done by ordinary Python functions.
The model decides *when* to call them and with *what* inputs (function calling).

In [ ]:
import math
import matplotlib.pyplot as plt
import io
import base64

def wcb_to_quadrantal(wcb_deg: float) -> dict:
    """Convert a whole circle bearing (0-360 degrees) to a quadrantal (reduced) bearing.

    Args:
        wcb_deg: Whole circle bearing in decimal degrees.
    """
    b = wcb_deg % 360
    if b <= 90:
        qb, ns, ew = b, "N", "E"
    elif b <= 180:
        qb, ns, ew = 180 - b, "S", "E"
    elif b <= 270:
        qb, ns, ew = b - 180, "S", "W"
    else:
        qb, ns, ew = 360 - b, "N", "W"
    return {"wcb_deg": b, "quadrantal_bearing": f"{ns} {qb:.4f} deg {ew}"}


def back_bearing(fore_bearing_deg: float) -> dict:
    """Back bearing (whole circle) of a line from its fore bearing.

    Args:
        fore_bearing_deg: Fore bearing (WCB) in decimal degrees.
    """
    return {"back_bearing_deg": round((fore_bearing_deg + 180) % 360, 4)}


def reduced_levels_hi_method(bm_rl: float, backsight: float, foresights: list[float]) -> dict:
    """Height of instrument method for ONE instrument setup.

    Args:
        bm_rl: Reduced level of the benchmark (m).
        backsight: Staff reading on the benchmark (m).
        foresights: Staff readings (intermediate/foresight) on the other points, in order (m).
    """
    hi = bm_rl + backsight
    return {"height_of_instrument": round(hi, 4),
            "reduced_levels": [round(hi - f, 4) for f in foresights]}


def rise_and_fall_method(first_rl: float, setups: list[list[float]]) -> dict:
    """Rise and fall method for a level book with one or more instrument setups.

    Args:
        first_rl: Reduced level of the first point (m).
        setups: One list per instrument setup. Each list holds the staff readings in order:
            the first is the backsight, the last is the foresight (any in between are intermediate).
            The foresight point of one setup is the backsight point of the next setup.
    """
    rl = first_rl
    rows = [{"reading": setups[0][0], "rise": None, "fall": None, "rl": round(rl, 4)}]
    for readings in setups:
        for prev, cur in zip(readings, readings[1:]):
            diff = prev - cur
            rl += diff
            rows.append({"reading": cur,
                         "rise": round(diff, 4) if diff > 0 else None,
                         "fall": round(-diff, 4) if diff < 0 else None,
                         "rl": round(rl, 4)})
    sum_bs = sum(s[0] for s in setups)
    sum_fs = sum(s[-1] for s in setups)
    return {"table": rows,
            "arithmetic_check": {"sum_BS_minus_sum_FS": round(sum_bs - sum_fs, 4),
                                 "last_RL_minus_first_RL": round(rl - first_rl, 4)}}


def curvature_refraction_correction(distance_km: float) -> dict:
    """Curvature, refraction and combined corrections for long sights in levelling.

    Args:
        distance_km: Length of sight in kilometres.
    """
    c = 0.0785 * distance_km ** 2
    r = c / 7
    return {"curvature_m": round(c, 5), "refraction_m": round(r, 5),
            "combined_m": round(0.0673 * distance_km ** 2, 5)}


def slope_correction(slope_length_m: float, height_difference_m: float) -> dict:
    """Tape correction for slope (exact formula).

    Args:
        slope_length_m: Measured length along the slope (m).
        height_difference_m: Difference in level between the two ends (m).
    """
    horizontal = math.sqrt(slope_length_m ** 2 - height_difference_m ** 2)
    return {"horizontal_length_m": round(horizontal, 4),
            "correction_m": round(slope_length_m - horizontal, 4)}


def stadia_tacheometry(staff_intercept_m: float, vertical_angle_deg: float,
                       multiplying_constant: float = 100.0, additive_constant: float = 0.0) -> dict:
    """Stadia tacheometry with an anallactic-less instrument and a VERTICAL staff.

    Horizontal distance D = k*s*cos^2(t) + C*cos(t); vertical component V = (k*s/2)*sin(2t) + C*sin(t).
    Then RL of staff station = RL of instrument axis + V - (middle-wire reading).

    Args:
        staff_intercept_m: Staff intercept s = top hair - bottom hair (m).
        vertical_angle_deg: Vertical angle t (positive for elevation, negative for depression).
        multiplying_constant: Stadia multiplying constant k (usually 100).
        additive_constant: Additive constant C (usually 0 for modern instruments).
    """
    t = math.radians(vertical_angle_deg)
    k, s, c = multiplying_constant, staff_intercept_m, additive_constant
    d = k * s * math.cos(t) ** 2 + c * math.cos(t)
    v = 0.5 * k * s * math.sin(2 * t) + c * math.sin(t)
    return {"horizontal_distance_m": round(d, 4), "vertical_component_m": round(v, 4)}


def interior_angle_check(interior_angles_deg: list[float]) -> dict:
    """Angular misclosure of a closed traverse from its measured interior angles.

    Args:
        interior_angles_deg: Measured interior angles in decimal degrees.
    """
    n = len(interior_angles_deg)
    expected = (2 * n - 4) * 90
    total = sum(interior_angles_deg)
    return {"n_stations": n, "expected_sum_deg": expected,
            "measured_sum_deg": round(total, 6), "misclosure_deg": round(total - expected, 6)}


def closed_traverse_bowditch(legs: list[list[float]], start_northing: float = 0.0,
                             start_easting: float = 0.0) -> dict:
    """Latitudes, departures, closing error and Bowditch adjustment for a closed traverse.

    Args:
        legs: One [length_m, whole_circle_bearing_deg] pair per traverse leg, in order around the traverse.
        start_northing: Northing (Y) of the starting station.
        start_easting: Easting (X) of the starting station.
    """
    lat = [L * math.cos(math.radians(b)) for L, b in legs]
    dep = [L * math.sin(math.radians(b)) for L, b in legs]
    perimeter = sum(L for L, _ in legs)
    sum_lat, sum_dep = sum(lat), sum(dep)
    e = math.hypot(sum_lat, sum_dep)
    adj_lat = [la - sum_lat * L / perimeter for la, (L, _) in zip(lat, legs)]
    adj_dep = [de - sum_dep * L / perimeter for de, (L, _) in zip(dep, legs)]
    n, east = start_northing, start_easting
    coords = [[round(n, 4), round(east, 4)]]
    for la, de in zip(adj_lat, adj_dep):
        n += la
        east += de
        coords.append([round(n, 4), round(east, 4)])
    return {"perimeter_m": round(perimeter, 4),
            "sum_latitudes": round(sum_lat, 4), "sum_departures": round(sum_dep, 4),
            "closing_error_m": round(e, 4),
            "relative_precision": f"1 in {round(perimeter / e)}" if e > 1e-9 else "perfect closure",
            "latitudes": [round(x, 4) for x in lat], "departures": [round(x, 4) for x in dep],
            "adjusted_latitudes": [round(x, 4) for x in adj_lat],
            "adjusted_departures": [round(x, 4) for x in adj_dep],
            "adjusted_coordinates_[N,E]": coords}


def polygon_area(points: list[list[float]]) -> dict:
    """Area of a polygon from coordinates (shoelace / coordinate method).

    Args:
        points: [[x1, y1], [x2, y2], ...] in order around the boundary (m).
    """
    n = len(points)
    s = 0.0
    for i in range(n):
        x1, y1 = points[i]
        x2, y2 = points[(i + 1) % n]
        s += x1 * y2 - x2 * y1
    a = abs(s) / 2
    return {"area_m2": round(a, 4), "area_hectare": round(a / 10000, 6)}


def simple_circular_curve(radius_m: float, deflection_angle_deg: float, chainage_of_pi_m: float) -> dict:
    """Elements of a simple circular curve.

    Args:
        radius_m: Radius R of the curve (m).
        deflection_angle_deg: Deflection angle (delta) between the straights (degrees).
        chainage_of_pi_m: Chainage of the point of intersection (m).
    """
    d = math.radians(deflection_angle_deg)
    T = radius_m * math.tan(d / 2)
    L = radius_m * d
    t1 = chainage_of_pi_m - T
    return {"tangent_length_m": round(T, 4), "curve_length_m": round(L, 4),
            "long_chord_m": round(2 * radius_m * math.sin(d / 2), 4),
            "external_distance_m": round(radius_m * (1 / math.cos(d / 2) - 1), 4),
            "mid_ordinate_m": round(radius_m * (1 - math.cos(d / 2)), 4),
            "chainage_T1_m": round(t1, 4), "chainage_T2_m": round(t1 + L, 4)}

def generate_matplotlib_plot(data_x: list[float], data_y: list[float], plot_type: str = 'line', title: str = '', xlabel: str = '', ylabel: str = '') -> str:
    """Generates a matplotlib plot as a base64 encoded PNG image string.

    Args:
        data_x: List of x-axis values.
        data_y: List of y-axis values.
        plot_type: Type of plot ('line', 'bar', 'scatter').
        title: Title of the plot.
        xlabel: Label for the x-axis.
        ylabel: Label for the y-axis.
    """
    plt.figure(figsize=(8, 5))
    if plot_type == 'line':
        plt.plot(data_x, data_y)
    elif plot_type == 'bar':
        plt.bar(data_x, data_y)
    elif plot_type == 'scatter':
        plt.scatter(data_x, data_y)
    else:
        plt.close()
        return "Unsupported plot type. Choose 'line', 'bar', or 'scatter'."

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close() # Close the plot to free memory
    image_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    return f'<img src="data:image/png;base64,{image_base64}" />' # Return as HTML img tag


TOOLS = [wcb_to_quadrantal, back_bearing, reduced_levels_hi_method, rise_and_fall_method,
         curvature_refraction_correction, slope_correction, stadia_tacheometry,
         interior_angle_check, closed_traverse_bowditch, polygon_area, simple_circular_curve,
         generate_matplotlib_plot]
print(len(TOOLS), "tools ready")

12 tools ready


## (Optional but recommended) Give SurveyGPT your course material
Upload the PDFs your professor allows (lecture notes, solved problems, question banks).
Text-based PDFs only – scanned images need OCR first. Skip this cell to run without notes.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the directory on Google Drive where your PDFs are stored
# IMPORTANT: Create a folder named 'SurveyGPT_Docs' in your Google Drive
# and place all your PDF files inside it.
DOCS_DIR = "/content/drive/MyDrive/SurveyGPT_Docs"

# Create the directory if it doesn't exist (though it should if Drive is mounted and folder exists)
os.makedirs(DOCS_DIR, exist_ok=True)

print(f"Google Drive mounted. Looking for PDFs in: {DOCS_DIR}")
if not os.path.exists(DOCS_DIR):
    print("Warning: The specified folder does not exist on your Google Drive. Please create it and place your PDFs inside.")
else:
    pdf_files = [f for f in os.listdir(DOCS_DIR) if f.lower().endswith('.pdf')]
    if pdf_files:
        print(f"Found {len(pdf_files)} PDF files in your Google Drive: {', '.join(pdf_files[:5])}{'...' if len(pdf_files) > 5 else ''}")
    else:
        print("No PDF files found in the specified Google Drive folder. Please upload them there.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted. Looking for PDFs in: /content/drive/MyDrive/SurveyGPT_Docs
Found 17 PDF files in your Google Drive: Group Test - Solution.pdf, Assignment 2_Solution.pdf, Compass Surveying.pdf, Leveling_For mid-semester.pdf, Assignment-1_Solution.pdf...


In [ ]:
import os

if os.path.exists(DOCS_DIR):
    pdf_files = [f for f in os.listdir(DOCS_DIR) if f.lower().endswith('.pdf')]
    if pdf_files:
        print(f"PDF files found in '{DOCS_DIR}':")
        for pdf_file in pdf_files:
            print(f"- {pdf_file}")
    else:
        print(f"No PDF files found in '{DOCS_DIR}'.")
else:
    print(f"The directory '{DOCS_DIR}' does not exist. Please ensure your Google Drive is mounted and the 'SurveyGPT_Docs' folder exists.")

PDF files found in '/content/drive/MyDrive/SurveyGPT_Docs':
- Group Test - Solution.pdf
- Assignment 2_Solution.pdf
- Compass Surveying.pdf
- Leveling_For mid-semester.pdf
- Assignment-1_Solution.pdf
- Plane Table Survey.pdf
- Levelling_2025.pdf
- Quiz-1-Solution.pdf
- Solution_Assignment_3.pdf
- Surveying Assignment_1.pdf
- Surveying Assignment_2.pdf
- Surveying_Assignment_1_Solution.pdf
- Surveying Assignment_3.pdf
- Surveying_Measurement and Errors.pdf
- Surveying_Introduction.pdf
- Tacheometry_2025.pdf
- Theodolite_2025.pdf


In [ ]:
import os, glob
import numpy as np
from pypdf import PdfReader

# The pip install commands are removed as they are causing conflicts and the necessary packages should already be installed.
# Ensure cell g4Rc2t3OWhtO is run before this cell to define DOCS_DIR.

from sentence_transformers import SentenceTransformer

# DOCS_DIR is now defined in the previous cell after mounting Google Drive
# os.makedirs(DOCS_DIR, exist_ok=True) # This is already handled by the Drive mounting cell

def load_chunks(chunk_size=900, overlap=150):
    """Read every PDF page and cut it into overlapping text chunks."""
    chunks = []
    for path in glob.glob(f"{DOCS_DIR}/*.pdf"):
        reader = PdfReader(path)
        for page_no, page in enumerate(reader.pages, start=1):
            text = (page.extract_text() or "").strip()
            start = 0
            while start < len(text):
                piece = text[start:start + chunk_size]
                if len(piece.strip()) > 50:
                    chunks.append({"text": piece,
                                   "source": f"{os.path.basename(path)} p.{page_no}"})
                start += chunk_size - overlap
    return chunks

chunks = load_chunks()
embedder, chunk_emb = None, None
if chunks:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")   # small, free, runs on Colab CPU
    chunk_emb = embedder.encode([c["text"] for c in chunks],
                                normalize_embeddings=True, show_progress_bar=True)
    print(f"Indexed {len(chunks)} chunks from your notes.")
else:
    print("No PDFs found - SurveyGPT will answer from the model's own knowledge + calculator tools.")

def retrieve(query, k=4, min_score=0.30):
    """Return the k most relevant note chunks (as text) and their sources."""
    if chunk_emb is None:
        return "", []
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_emb @ q                      # cosine similarity (vectors are normalised)
    top = [i for i in np.argsort(-scores)[:k] if scores[i] >= min_score]
    context = "\n\n".join(f"[{chunks[i]['source']}]\n{chunks[i]['text']}" for i in top)
    sources = sorted({chunks[i]["source"] for i in top})
    return context, sources

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Indexed 305 chunks from your notes.


In [ ]:
pip install numpy==1.24.4

  Using cached numpy-1.24.4.tar.gz (10.9 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
pip install --upgrade sentence-transformers

After installing the correct `numpy` version and upgrading `sentence-transformers`, please rerun the cell above (cell `vfPw4O9ZWhtP`) to correctly load the chunks and embedder.

In [ ]:
from google import genai
from google.genai import types

MODEL = "gemini-3.5-flash"     # if you get a "model not found" error, see the tip in the notes below

client = genai.Client(api_key=GEMINI_API_KEY)

SYSTEM_PROMPT = """You are SurveyGPT, a teaching assistant for the Surveying course of a B.Tech Civil Engineering programme.

Scope: introduction & classification of surveys, chain and tape surveying, compass surveying and local attraction,
plane table surveying, levelling (HI and rise-and-fall, reciprocal levelling, curvature and refraction, contouring),
theodolite surveying, traverse computation (Bowditch/transit), tacheometry, simple/compound/reverse/transition curves,
areas and volumes, trilateration/triangulation basics, EDM/total station, GPS/GNSS and remote sensing basics.

How to answer:
1. For numerical problems: list the given data, state the formula, substitute with units, then give the final answer.
   Use the provided calculator tools for the arithmetic instead of computing long calculations yourself.
   You can also generate plots and diagrams using the `generate_matplotlib_plot` tool when appropriate or requested,
   providing lists of x and y data, the plot type (line, bar, scatter), title, and axis labels.
2. For theory questions: give a clear, exam-oriented explanation with a short example or a sketch description.
3. If course-material excerpts are supplied, prefer them and mention which source you used.
4. If the question is ambiguous or data is missing, ask for exactly what is missing instead of guessing.
5. Use metric units and standard Indian textbook conventions (WCB, RL, chainage). If unsure, say so - never invent facts.
6. Stay within surveying/civil engineering; politely decline unrelated requests.
7.Never use LaTeX or markdown math syntax (no $...$, \\frac{}{}, \\Delta, \\times, ^{}, _{}, etc.) - the chat window
   cannot render it and it shows up as broken text. Write every formula in plain text with normal keyboard characters
   instead, e.g. "R * tan(delta / 2)" or "sqrt(x)", and spell out Greek letters (delta, theta) rather than using symbols."""

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    tools=TOOLS,            # Python functions -> the SDK turns them into callable tools automatically
    temperature=0.2,        # low = more consistent, fewer creative errors
)
chat = client.chats.create(model=MODEL, config=config)   # keeps conversation memory
print("Model ready:", MODEL)

Model ready: gemini-3.5-flash


In [ ]:
import gradio as gr
from PIL import Image # Import Pillow for image processing
import pytesseract # Import pytesseract for OCR

def respond(message, history):
    text_message = message.get('text', '') if isinstance(message, dict) else message
    image_files = message.get('files', []) if isinstance(message, dict) else []

    ocr_text = []
    if image_files:
        print(f"DEBUG: Received {len(image_files)} image files.")
        for img_path in image_files:
            print(f"DEBUG: Processing image path: {img_path}")
            try:
                img = Image.open(img_path)
                extracted_text = pytesseract.image_to_string(img)
                print(f"DEBUG: OCR extracted text (first 100 chars): {extracted_text[:100]}...")
                if extracted_text.strip():
                    ocr_text.append(extracted_text)
            except Exception as e:
                print(f"ERROR: Error processing image {img_path}: {e}")

    combined_message = text_message
    if ocr_text:
        combined_message += "\n\n[OCR from image]:\n" + "\n".join(ocr_text)

    print(f"DEBUG: Combined message for LLM (first 200 chars): {combined_message[:200]}...")

    if not combined_message.strip():
        return "Please provide a question, either by typing or by uploading an image."

    context, sources = retrieve(combined_message)
    prompt = (f"Course material excerpts (use if relevant):\n{context}\n\n---\nStudent question: {combined_message}"
              if context else combined_message)
    try:
        full_response = chat.send_message(prompt)
        reply_parts = []
        for part in full_response.candidates[0].content.parts:
            if part.text:
                reply_parts.append(part.text)
            elif part.function_call:
                tool_name = part.function_call.name
                tool_args = {k: v for k, v in part.function_call.args.items()}

                if tool_name == 'generate_matplotlib_plot':
                    try:
                        plot_html = generate_matplotlib_plot(**tool_args)
                        reply_parts.append(plot_html)
                    except Exception as tool_e:
                        reply_parts.append(f"Error executing plotting tool: {tool_e}")
                else:
                    reply_parts.append(f"Tool call: {tool_name} with args {tool_args}")

        reply = "\n".join(reply_parts)
        if not reply.strip():
            reply = "I could not produce an answer - please rephrase or try again."
    except Exception as e:
        # This error is critical, so we return it directly.
        return f"Error while contacting the model: {e}"

    # Debugging prints - these will appear in the Colab output, not Gradio.
    print(f"DEBUG: Final reply length: {len(reply)}")
    print(f"DEBUG: Final reply (first 200 chars): {reply[:200]}")

    return reply

demo = gr.ChatInterface(
    respond,
    title="SurveyGPT - Surveying Assistant",
    description="Ask theory questions or paste numerical problems (levelling, traverse, curves, tacheometry...).",
    examples=[
        "Explain the difference between rise and fall method and HI method.",
        "Convert WCB 236 30' to quadrantal bearing and find its back bearing.",
        "A simple circular curve has R = 300 m, deflection angle 60 deg and PI chainage 1250 m. Find all curve elements.",
        "Compute the closing error and Bowditch adjustment: AB 100 m 0 deg, BC 80 m 90 deg, CD 100 m 181 deg, DA 80 m 270 deg.",
    ],
    multimodal=True # Enable multimodal input
)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://36e5ec64f8d1abfc32.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


DEBUG: Received 1 image files.
DEBUG: Processing image path: /tmp/gradio/9577bbfce7f0231437f4f378f7baceae94195a2ad38bfc9b4ec37eb4184cf2c6/WhatsApp Image 2026-09-25 at 3.14.09 PM.jpeg
DEBUG: OCR extracted text (first 100 chars): oss

Indian Institute of Technology Bhubaneswar
School of Infrastructure
Mic-Semester Examination - ...
DEBUG: Combined message for LLM (first 200 chars): 1,2,3 answers

[OCR from image]:
oss

Indian Institute of Technology Bhubaneswar
School of Infrastructure
Mic-Semester Examination - Autumn - 2026

Subject: Geomatics Practices Code: CE2L105
Date: 18 ...
DEBUG: Received 1 image files.
DEBUG: Processing image path: {'path': '/tmp/gradio/9577bbfce7f0231437f4f378f7baceae94195a2ad38bfc9b4ec37eb4184cf2c6/WhatsApp Image 2026-09-25 at 3.14.09 PM.jpeg', 'url': '/gradio_api/file=/tmp/gradio/9577bbfce7f0231437f4f378f7baceae94195a2ad38bfc9b4ec37eb4184cf2c6/WhatsApp%20Image%202026-09-25%20at%203.14.09%20PM.jpeg', 'size': None, 'orig_name': None, 'mime_type': 'image/jpeg',

KeyboardInterrupt: 

### Tips
* **Model not found / quota error?** Run `for m in client.models.list(): print(m.name)` to see the models your key can use, then change `MODEL`.
* **Colab resets** when the runtime disconnects – re-upload PDFs (or mount Google Drive) and re-run the cells.
* **All visitors of the public link share one conversation** (single `chat` object). Fine for a demo; for real class use, keep one chat per user.
* Always verify important answers against your textbook – LLMs can make mistakes.